In [1]:
%pip install nltk pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 16.0 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.3.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import re
import pandas as pd

df = pd.read_csv("../data/comments_raw.csv")
labels = pd.read_csv("../data/labels_manual.csv")
df = df.merge(labels, on="id", how="left")

NEG = {"không", "ko", "k", "chẳng", "chả", "chưa"}

def tokenize(text):
    tokens, neg = [], False
    for t in re.findall(r"\w+|[.,!?;]", text.lower()):
        if t in ".,!?;":
            neg = False
        else:
            tokens.append("NOT_" + t if neg else t)
            if t in NEG:
                neg = True
    return tokens

df["tokens"] = df["text"].apply(tokenize)
print(df["label"].value_counts(dropna=False))

label
NaN    180
pos     10
neg     10
Name: count, dtype: int64


In [3]:
import nltk

def features(tokens):
    return {f"contains({w})": True for w in set(tokens)}

labeled = df[df["label"].notna()]
train_set = [(features(t), l) for t, l in zip(labeled["tokens"], labeled["label"])]

classifier = nltk.NaiveBayesClassifier.train(train_set)
classifier.show_most_informative_features(10)

for s in ["claude đỉnh thật", "gemini phế"]:
    p = classifier.prob_classify(features(tokenize(s)))
    print(s, "->", p.max(), round(p.prob(p.max()), 2))

Most Informative Features
          contains(luôn) = True              pos : neg    =      3.0 : 1.0
            contains(là) = True              pos : neg    =      3.0 : 1.0
           contains(tâm) = True              pos : neg    =      2.3 : 1.0
        contains(NOT_có) = True              neg : pos    =      1.7 : 1.0
           contains(chứ) = True              neg : pos    =      1.7 : 1.0
            contains(mà) = True              pos : neg    =      1.7 : 1.0
           contains(mới) = True              neg : pos    =      1.7 : 1.0
         contains(người) = True              pos : neg    =      1.7 : 1.0
          contains(nhất) = True              pos : neg    =      1.7 : 1.0
           contains(thứ) = True              pos : neg    =      1.7 : 1.0
claude đỉnh thật -> pos 0.94
gemini phế -> neg 0.75


In [4]:
THRESHOLD = 0.9
df["pred"] = df["label"]                    
df["conf"] = None                         
df["round"] = None                           
df.loc[df["label"].notna(), "round"] = 0   

r = 0
while True:
    r += 1
    known = df[df["pred"].notna()]
    classifier = nltk.NaiveBayesClassifier.train(
        [(features(t), l) for t, l in zip(known["tokens"], known["pred"])])
    added = 0
    for i in df.index[df["pred"].isna()]:
        p = classifier.prob_classify(features(df.at[i, "tokens"]))
        if p.prob(p.max()) >= THRESHOLD:
            df.at[i, "pred"] = p.max()
            df.at[i, "conf"] = round(p.prob(p.max()), 2)
            df.at[i, "round"] = r
            added += 1
    print(f"Vong {r}: them {added} cau, tong co nhan {df['pred'].notna().sum()}")
    if added == 0:
        break

print(df["pred"].value_counts(dropna=False))

Vong 1: them 84 cau, tong co nhan 104
Vong 2: them 62 cau, tong co nhan 166
Vong 3: them 8 cau, tong co nhan 174
Vong 4: them 5 cau, tong co nhan 179
Vong 5: them 0 cau, tong co nhan 179
pred
neg    115
pos     64
NaN     21
Name: count, dtype: int64
